In [2]:
import mediapipe as mp
import cv2
import numpy as np
from collections import deque, Counter

# Pose Landmarker 설정
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE
)

# ── 핵심 인덱스 ──────────────────────────────
NOSE           = 0
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12

# ── 임계값 (튜닝 가능) ───────────────────────
HEAD_MOVE_THRESHOLD     = 0.05
BODY_MOVE_THRESHOLD     = 0.05
SHOULDER_TILT_THRESHOLD = 0.04
PENALTY_PER_VIOLATION   = 5
MAX_SCORE               = 100

class PostureAnalyzer:
    def __init__(self, history_len=5):
        self.head_history  = deque(maxlen=history_len)
        self.body_history  = deque(maxlen=history_len)
        self.violations    = []
        self.frame_idx     = 0

    def _extract_keypoints(self, landmarks):
        nose  = (landmarks[NOSE].x, landmarks[NOSE].y)
        l_sho = (landmarks[LEFT_SHOULDER].x,  landmarks[LEFT_SHOULDER].y)
        r_sho = (landmarks[RIGHT_SHOULDER].x, landmarks[RIGHT_SHOULDER].y)
        body_center   = ((l_sho[0]+r_sho[0])/2, (l_sho[1]+r_sho[1])/2)
        shoulder_tilt = abs(l_sho[1] - r_sho[1])
        return nose, body_center, shoulder_tilt

    def update(self, landmarks):
        nose, body_center, shoulder_tilt = self._extract_keypoints(landmarks)
        head_move = 0.0
        body_move = 0.0

        if self.head_history:
            prev = self.head_history[-1]
            head_move = np.hypot(nose[0]-prev[0], nose[1]-prev[1])

        if self.body_history:
            prev = self.body_history[-1]
            body_move = np.hypot(body_center[0]-prev[0], body_center[1]-prev[1])

        self.head_history.append(nose)
        self.body_history.append(body_center)

        if head_move > HEAD_MOVE_THRESHOLD:
            self.violations.append((self.frame_idx, '머리', head_move))
        if body_move > BODY_MOVE_THRESHOLD:
            self.violations.append((self.frame_idx, '상반신', body_move))
        if shoulder_tilt > SHOULDER_TILT_THRESHOLD:
            self.violations.append((self.frame_idx, '어깨기울기', shoulder_tilt))

        self.frame_idx += 1

    def get_final_score(self):
        total_penalty = len(self.violations) * PENALTY_PER_VIOLATION
        score = max(0, MAX_SCORE - total_penalty)
        type_count = Counter(v[1] for v in self.violations)

        feedback = []
        if type_count.get('머리', 0) > 3:
            feedback.append(f"머리 움직임이 {type_count['머리']}회 감지. 시선을 고정하세요.")
        if type_count.get('상반신', 0) > 3:
            feedback.append(f"상반신 움직임이 {type_count['상반신']}회 감지. 자세를 안정적으로 유지하세요.")
        if type_count.get('어깨기울기', 0) > 3:
            feedback.append(f"어깨 기울어짐이 {type_count['어깨기울기']}회 감지. 수평을 유지하세요.")
        if not feedback:
            feedback.append("자세가 매우 안정적이었습니다!")

        return {'score': score, 'violations': dict(type_count), 'feedback': feedback}


# ── 동영상 분석 함수 ──────────────────────────
def analyze_video(video_path, stride=5):
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"❌ 동영상을 열 수 없습니다: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    duration     = total_frames / fps if fps > 0 else 0
    print(f"📹 동영상 정보: {total_frames}프레임 / {fps:.1f}fps / {duration:.1f}초")

    analyzer    = PostureAnalyzer(history_len=5)
    detected    = 0
    undetected  = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        frame_num = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_num % stride != 0:
                frame_num += 1
                continue
            

            # 매 프레임마다 분석 (필요시 stride 조절 가능)
            img_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result   = landmarker.detect(mp_image)

            if result.pose_landmarks:
                analyzer.update(result.pose_landmarks[0])
                detected += 1
            else:
                undetected += 1

            frame_num += 1
            if frame_num % 100 == 0:
                print(f"  처리 중... {frame_num}/{total_frames} 프레임")

    cap.release()

    # ── 결과 출력 ─────────────────────────────
    result = analyzer.get_final_score()
    type_count = result['violations']

    print("\n" + "="*40)
    print("        📊 자세 분석 결과")
    print("="*40)
    print(f"분석 프레임:     {detected}프레임 (미감지: {undetected}프레임)")
    print(f"머리 과다 움직임:  {type_count.get('머리', 0)}회")
    print(f"상반신 과다 움직임: {type_count.get('상반신', 0)}회")
    print(f"어깨 기울어짐:    {type_count.get('어깨기울기', 0)}회")
    print(f"총 위반 횟수:     {sum(type_count.values())}회")
    print(f"\n최종 점수:       {result['score']} / 100")
    print("-"*40)
    for fb in result['feedback']:
        print(f"  → {fb}")
    print("="*40)


# ── 실행 ─────────────────────────────────────
VIDEO_PATH = '얼굴움직임.mp4'   # ← 동영상 파일명으로 변경
analyze_video(VIDEO_PATH)

📹 동영상 정보: 609프레임 / 30.1fps / 20.2초


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773025344.831101   12974 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773025344.886408   12974 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773025345.021106   12997 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.



        📊 자세 분석 결과
분석 프레임:     121프레임 (미감지: 1프레임)
머리 과다 움직임:  22회
상반신 과다 움직임: 5회
어깨 기울어짐:    36회
총 위반 횟수:     63회

최종 점수:       0 / 100
----------------------------------------
  → 머리 움직임이 22회 감지. 시선을 고정하세요.
  → 상반신 움직임이 5회 감지. 자세를 안정적으로 유지하세요.
  → 어깨 기울어짐이 36회 감지. 수평을 유지하세요.
